# Exercise 5 — RetrievalAgent

**RetrievalAgent** runs the agentic retrieval loop: at each step it decides to retrieve or answer, accumulates docs across iterations, and falls back to a final LLM call when `max_iterations` is reached.  Same four-method class shape as all prior Section 6 agents.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

class SimpleRetriever:
    def __init__(self):
        self._docs = []
    def add(self, doc):
        self._docs.append(doc); return self
    def add_all(self, docs):
        for d in docs: self._docs.append(d)
        return self
    def _score(self, query, doc):
        q = set(query.lower().split())
        d = set(doc.content.lower().split())
        return len(q & d) / (len(q | d) + 1e-9)
    def search(self, query, top_k=3):
        if not self._docs: return []
        return sorted(self._docs, key=lambda doc: self._score(query, doc), reverse=True)[:top_k]
    def __len__(self): return len(self._docs)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
def format_docs(docs):
    if not docs:
        return "No documents found."
    lines = []
    for i, d in enumerate(docs, 1):
        source = d.metadata.get("source", f"doc{i}")
        lines.append(f"[{i}] ({source}) {d.content}")
    return "\n".join(lines)

def build_retrieval_prompt(question, docs):
    context = format_docs(docs)
    system = "\n".join([
        "You are a helpful assistant.",
        "Answer the question using ONLY the provided documents.",
        "If the answer is not in the documents, say: I don't have enough information.",
        "Cite document numbers like [1] when referencing specific facts.",
    ])
    user = "Documents:\n" + context + "\n\nQuestion: " + str(question)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def retrieve_and_answer(question, retriever, top_k=3, llm_fn=None):
    docs = retriever.search(question, top_k=top_k)
    prompt = build_retrieval_prompt(question, docs)
    answer = call_llm(prompt, llm_fn=llm_fn)
    return {"question": question, "docs": docs, "answer": answer}
def build_agent_step_prompt(question, context):
    has_context = bool(context) and context != "No documents found."
    ctx_line = "Current context:\n" + context if has_context else "No context retrieved yet."
    system = "\n".join([
        "You are a research agent. Decide your next action.",
        'To search: {"action": "retrieve", "query": "your search query"}',
        'To answer: {"action": "answer",   "text":  "your final answer"}',
        "Use retrieve to gather information; use answer when you have enough.",
        "Reply with ONLY valid JSON.",
    ])
    user = "Question: " + str(question) + "\n\n" + ctx_line
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def parse_agent_action(text):
    data = safe_parse_json(text) or {}
    if data.get("action") == "answer":
        return {"action": "answer", "text": str(data.get("text", ""))}
    query = str(data.get("query", ""))
    return {"action": "retrieve", "query": query}
def _mock_retrieval_llm(script):
    idx = [0]
    def _fn(messages):
        i = min(idx[0], len(script) - 1)
        idx[0] += 1
        return json.dumps(script[i])
    return _fn

# ── Exercise: implement RetrievalAgent ───────────────────────────────────────

class RetrievalAgent:
    """An agent that iteratively retrieves documents and answers questions."""

    def __init__(self, retriever, llm_fn=None, top_k=3, max_iterations=5):
        self.retriever = retriever
        self._llm_fn = llm_fn
        self._top_k = top_k
        self.max_iterations = max_iterations   # required: prevents infinite loops
        self._history = []

    def ask(self, question):
        # TODO: run the agentic loop:
        # - all_docs = [], steps = []
        # - for iteration in range(self.max_iterations):
        #     context = format_docs(all_docs) or "No documents retrieved yet."
        #     prompt = build_agent_step_prompt(question, context)
        #     response = call_llm(prompt, llm_fn=self._llm_fn)
        #     act = parse_agent_action(response)
        #     if act["action"] == "answer": record + return
        #     else: search retriever, extend all_docs, append step
        # - if loop exhausts: build_retrieval_prompt -> call_llm -> record + return
        return {}

    def history(self):
        # TODO: return a copy of self._history
        return []

    def clear_history(self):
        # TODO: clear self._history
        pass


### Checks

In [ ]:
checks = 0

retriever = SimpleRetriever()
retriever.add_all([
    Document("Python is a high-level programming language.", {"source": "intro"}),
    Document("Python was created by Guido van Rossum in 1991.", {"source": "history"}),
])

# 1 — RetrievalAgent constructs with max_iterations
try:
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm([{"action": "answer", "text": "x"}]))
    assert agent.max_iterations == 5
    checks += 1; print("✅ 1 RetrievalAgent constructs with default max_iterations=5")
except Exception as e:
    print("❌ 1:", e)

# 2 — ask() returns dict with question, docs, answer, steps
try:
    script = [
        {"action": "retrieve", "query": "Python"},
        {"action": "answer",   "text":  "Python is a programming language."},
    ]
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm(script))
    r = agent.ask("What is Python?")
    assert "question" in r and "docs" in r and "answer" in r and "steps" in r
    checks += 1; print("✅ 2 ask() returns {question, docs, answer, steps}")
except Exception as e:
    print("❌ 2:", e)

# 3 — answer is returned from the answer action
try:
    script = [{"action": "answer", "text": "Python is great."}]
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm(script))
    r = agent.ask("Tell me about Python")
    assert r["answer"] == "Python is great."
    checks += 1; print("✅ 3 answer from 'answer' action is returned correctly")
except Exception as e:
    print("❌ 3:", e)

# 4 — retrieve step extends docs and is recorded
try:
    script = [
        {"action": "retrieve", "query": "Python"},
        {"action": "answer",   "text":  "Done."},
    ]
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm(script))
    r = agent.ask("Python?")
    retrieve_steps = [s for s in r["steps"] if s["action"] == "retrieve"]
    assert len(retrieve_steps) >= 1
    assert len(r["docs"]) > 0
    checks += 1; print("✅ 4 retrieve step recorded and docs accumulated")
except Exception as e:
    print("❌ 4:", e)

# 5 — history grows with each ask
try:
    script = [{"action": "answer", "text": "ok"}]
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm(script))
    agent.ask("q1"); agent.ask("q2")
    assert len(agent.history()) == 2
    checks += 1; print("✅ 5 history() grows with each ask()")
except Exception as e:
    print("❌ 5:", e)

# 6 — clear_history empties history
try:
    script = [{"action": "answer", "text": "ok"}]
    agent = RetrievalAgent(retriever, llm_fn=_mock_retrieval_llm(script))
    agent.ask("q")
    agent.clear_history()
    assert agent.history() == []
    checks += 1; print("✅ 6 clear_history() empties history")
except Exception as e:
    print("❌ 6:", e)

print(f"\n{checks}/6 checks passed!")
